1. Ingest sample order data into a Spark DataFrame.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
import random
import time
from datetime import datetime, timedelta
import uuid

spark = SparkSession.builder.getOrCreate()
TOTAL_ROWS = 10000

# Date range for random timestamps
start_date = datetime(2025, 10, 1)
end_date   = datetime(2025, 11, 15)

# Configurations
country_list = ["US", "IN", "UK", "JP", "AU", "DE", "SG"]
currency_map = {
    "US": "USD", "IN": "INR", "UK": "GBP",
    "JP": "JPY", "AU": "AUD", "DE": "EUR", "SG": "SGD"
}
order_status = ["CREATED", "PAID", "CANCELLED"]


def get_random_timestamp(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)


order_rows = []

for i in range(TOTAL_ROWS):
    oid = str(uuid.uuid4())
    cust = f"CUST-{random.randint(1000, 9999)}"
    country = random.choice(country_list)
    ts = get_random_timestamp(start_date, end_date)
    amount = round(random.uniform(5.0, 800.0), 2)
    curr = currency_map[country]

    # Weighted statuses (different logic)
    status = random.choices(
        population=order_status,
        weights=[0.15, 0.75, 0.10],  # CREATED, PAID, CANCELLED
        k=1
    )[0]

    order_rows.append((
        oid, ts, cust, country, amount, curr, status
    ))


order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True)
])


df_orders = spark.createDataFrame(order_rows, schema=order_schema)

display(df_orders.limit(5))


order_id,order_timestamp,customer_id,country,amount,currency,status
469d36fd-25a4-4a3e-85c0-147a90a46cbd,2025-11-07T00:44:41.000Z,CUST-4618,DE,470.48,EUR,CANCELLED
6148b846-b335-4003-a8a7-9953cb437e54,2025-10-11T21:26:54.000Z,CUST-3598,DE,723.55,EUR,PAID
7ab49dbb-30ab-4539-9bea-713c0f023fca,2025-10-07T03:19:44.000Z,CUST-9955,US,524.63,USD,PAID
8f1cb5d0-ac34-4ee3-9b60-c881d8ed6f51,2025-11-02T23:16:30.000Z,CUST-9601,IN,149.15,INR,PAID
95fef143-5d0d-44fb-9ac8-754f841b4476,2025-10-15T03:22:36.000Z,CUST-2558,AU,668.03,AUD,PAID


In [0]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)



2. Add a derived column order_date (date only from order_timestamp).

In [0]:
from pyspark.sql.functions import to_date

# Add derived date column
df_orders_with_date = df_orders.withColumn(
    "order_date",
    to_date("order_timestamp")
)

# Peek at the result
display(df_orders_with_date.limit(5))
df_orders_with_date.printSchema()

order_id,order_timestamp,customer_id,country,amount,currency,status,order_date
469d36fd-25a4-4a3e-85c0-147a90a46cbd,2025-11-07T00:44:41.000Z,CUST-4618,DE,470.48,EUR,CANCELLED,2025-11-07
6148b846-b335-4003-a8a7-9953cb437e54,2025-10-11T21:26:54.000Z,CUST-3598,DE,723.55,EUR,PAID,2025-10-11
7ab49dbb-30ab-4539-9bea-713c0f023fca,2025-10-07T03:19:44.000Z,CUST-9955,US,524.63,USD,PAID,2025-10-07
8f1cb5d0-ac34-4ee3-9b60-c881d8ed6f51,2025-11-02T23:16:30.000Z,CUST-9601,IN,149.15,INR,PAID,2025-11-02
95fef143-5d0d-44fb-9ac8-754f841b4476,2025-10-15T03:22:36.000Z,CUST-2558,AU,668.03,AUD,PAID,2025-10-15


root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)



3. Write the DataFrame as a Delta table partitioned by country and order_date.

In [0]:
df_orders = df_orders.withColumn("order_date", to_date("order_timestamp"))

In [0]:
output_path = "/Volumes/main/default/orders_delta"   

(
    df_orders
        .write
        .format("delta")
        .mode("overwrite")              
        .partitionBy("country", "order_date")
        .save(output_path)
)

print("Delta table written successfully with partitions!")


Delta table written successfully with partitions!


4. Verify the partition structure in the storage path.

In [0]:
display(dbutils.fs.ls("/Volumes/main/default/orders_delta"))

path,name,size,modificationTime
dbfs:/Volumes/main/default/orders_delta/_delta_log/,_delta_log/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=AU/,country=AU/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=DE/,country=DE/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=IN/,country=IN/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=JP/,country=JP/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=SG/,country=SG/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=UK/,country=UK/,0,1764611782988
dbfs:/Volumes/main/default/orders_delta/country=US/,country=US/,0,1764611782988


In [0]:
display(dbutils.fs.ls("/Volumes/main/default/orders_delta/country=US"))

path,name,size,modificationTime
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-01/,order_date=2025-10-01/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-02/,order_date=2025-10-02/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-03/,order_date=2025-10-03/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-04/,order_date=2025-10-04/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-05/,order_date=2025-10-05/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-06/,order_date=2025-10-06/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-07/,order_date=2025-10-07/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-08/,order_date=2025-10-08/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-09/,order_date=2025-10-09/,0,1764611793205
dbfs:/Volumes/main/default/orders_delta/country=US/order_date=2025-10-10/,order_date=2025-10-10/,0,1764611793205


5. Run queries that demonstrate partition pruning (e.g., filter on a single country
and/or date).

In [0]:
#Query by Country
df_us = spark.read.format("delta").load("/Volumes/main/default/orders_delta") \
  .filter("country = 'US'")

df_us.show(5)

+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|            order_id|    order_timestamp|customer_id|country|amount|currency|   status|order_date|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|0ce692f2-13ca-46f...|2025-11-08 11:59:52|  CUST-9506|     US|757.76|     USD|CANCELLED|2025-11-08|
|d2916319-771d-40e...|2025-11-08 14:31:23|  CUST-8980|     US|124.97|     USD|     PAID|2025-11-08|
|9b05daae-6fc3-45a...|2025-11-08 18:33:42|  CUST-5230|     US|436.96|     USD|     PAID|2025-11-08|
|035eaa3c-2aa2-4cb...|2025-11-08 09:11:25|  CUST-1073|     US|652.49|     USD|     PAID|2025-11-08|
|20002243-fbae-4b3...|2025-11-08 05:23:42|  CUST-8488|     US|292.48|     USD|  CREATED|2025-11-08|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
only showing top 5 rows


In [0]:
#Query by Country + Date 
df_us_date = (
    spark.read.format("delta")
         .load("/Volumes/main/default/orders_delta")
         .filter("country = 'IN' AND order_date = '2025-10-10'")
)

df_us_date.show()

+--------------------+-------------------+-----------+-------+------+--------+-------+----------+
|            order_id|    order_timestamp|customer_id|country|amount|currency| status|order_date|
+--------------------+-------------------+-----------+-------+------+--------+-------+----------+
|703c528f-3bf0-484...|2025-10-10 15:17:25|  CUST-3598|     IN|456.43|     INR|   PAID|2025-10-10|
|7467e181-bfd3-479...|2025-10-10 23:02:24|  CUST-8274|     IN|255.83|     INR|   PAID|2025-10-10|
|9bc4a721-f336-401...|2025-10-10 17:10:09|  CUST-7420|     IN|609.29|     INR|   PAID|2025-10-10|
|963060c2-0bcb-442...|2025-10-10 13:28:44|  CUST-8437|     IN|231.88|     INR|   PAID|2025-10-10|
|1103b7e3-43e7-461...|2025-10-10 23:16:53|  CUST-4890|     IN|204.29|     INR|CREATED|2025-10-10|
|b8e5e288-8354-46b...|2025-10-10 12:16:03|  CUST-8921|     IN|674.83|     INR|   PAID|2025-10-10|
|ef71b7ad-36a5-4c7...|2025-10-10 17:56:13|  CUST-6524|     IN|518.05|     INR|   PAID|2025-10-10|
|b08a28f1-44bd-412..

In [0]:
#Count by country (pruning happens)
df = spark.read.format("delta").load("/Volumes/main/default/orders_delta")

df.filter("country = 'UK'").count()

1355

In [0]:
#Select orders for a date range — automatic pruning
df_range = (
    df.filter("order_date >= '2025-10-05' AND order_date <= '2025-10-12'")
)

df_range.show(10)

+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|            order_id|    order_timestamp|customer_id|country|amount|currency|   status|order_date|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|9e27c3b1-dc1d-4f3...|2025-10-09 13:52:06|  CUST-8880|     JP|296.96|     JPY|  CREATED|2025-10-09|
|ac735668-de5b-42e...|2025-10-09 07:23:55|  CUST-4858|     JP|482.66|     JPY|     PAID|2025-10-09|
|c01deb20-3ad2-42f...|2025-10-09 13:38:08|  CUST-5267|     JP| 609.6|     JPY|  CREATED|2025-10-09|
|822b4a19-882e-4dd...|2025-10-09 05:46:01|  CUST-1636|     JP|354.24|     JPY|CANCELLED|2025-10-09|
|bd489906-c3c9-43a...|2025-10-09 02:04:53|  CUST-9847|     JP|  44.7|     JPY|     PAID|2025-10-09|
|2773d043-7142-400...|2025-10-09 02:11:11|  CUST-6274|     JP|525.81|     JPY|     PAID|2025-10-09|
|8bacd2a1-4783-4a2...|2025-10-09 08:19:18|  CUST-7547|     JP|556.34|     JPY|CANCELLED|2025-10-09|


In [0]:
#Aggregation on a single partition
df_in_agg = (
    df.filter("country = 'IN' AND order_date = '2025-10-15'")
      .groupBy("status")
      .count()
)

df_in_agg.show()

+---------+-----+
|   status|count|
+---------+-----+
|CANCELLED|    5|
|  CREATED|    3|
|     PAID|   27|
+---------+-----+



In [0]:
(
    df.filter("country = 'US'")
      .select("order_id")
      .explain(True)
)

== Parsed Logical Plan ==
'Project ['order_id]
+- 'Filter ('country = US)
   +- Relation [order_id#11471,order_timestamp#11472,customer_id#11473,country#11474,amount#11475,currency#11476,status#11477,order_date#11478] parquet

== Analyzed Logical Plan ==
order_id: string
Project [order_id#11471]
+- Filter (country#11474 = US)
   +- Relation [order_id#11471,order_timestamp#11472,customer_id#11473,country#11474,amount#11475,currency#11476,status#11477,order_date#11478] parquet

== Optimized Logical Plan ==
Project [order_id#11471]
+- Filter (isnotnull(country#11474) AND (country#11474 = US))
   +- Relation [order_id#11471,order_timestamp#11472,customer_id#11473,country#11474,amount#11475,currency#11476,status#11477,order_date#11478] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonProject [order_id#11471]
      +- PhotonScan parquet [order_id#11471,country#11474,order_date#11478] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDe

6. Demonstrate Delta Lake Time Travel:

o Write data, update some rows, then query older versions.

In [0]:
#Update Some Rows (create a new version)
from delta.tables import DeltaTable

table_path = "/Volumes/main/default/orders_delta"

delta_tbl = DeltaTable.forPath(spark, table_path)

# Update some data to create a new version
delta_tbl.update(
    condition="country = 'US'",
    set={"status": "'CANCELLED'"}
)

print("Update complete — new table version created.")

Update complete — new table version created.


In [0]:
#View Delta Table History
history_df = delta_tbl.history()
history_df.show(truncate=False)

+-------+-------------------+--------------+-------------------------+---------+----------------------------------------------------------------------------------+----+------------------+------------------------+-----------+-----------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                 |operation|operationParameters                                                               |job |notebook          |clusterId               |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                

In [0]:
#Time Travel: Read Version 0 (Before Update)
df_old = spark.read.format("delta").option("versionAsOf", 0).load(table_path)
df_old.filter("country = 'US'").show(5)

+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|            order_id|    order_timestamp|customer_id|country|amount|currency|   status|order_date|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|0ce692f2-13ca-46f...|2025-11-08 11:59:52|  CUST-9506|     US|757.76|     USD|CANCELLED|2025-11-08|
|d2916319-771d-40e...|2025-11-08 14:31:23|  CUST-8980|     US|124.97|     USD|     PAID|2025-11-08|
|9b05daae-6fc3-45a...|2025-11-08 18:33:42|  CUST-5230|     US|436.96|     USD|     PAID|2025-11-08|
|035eaa3c-2aa2-4cb...|2025-11-08 09:11:25|  CUST-1073|     US|652.49|     USD|     PAID|2025-11-08|
|20002243-fbae-4b3...|2025-11-08 05:23:42|  CUST-8488|     US|292.48|     USD|  CREATED|2025-11-08|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
only showing top 5 rows


In [0]:
#Read Latest Version (after update)
df_latest = spark.read.format("delta").load(table_path)
df_latest.filter("country = 'US'").show(5)

+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|            order_id|    order_timestamp|customer_id|country|amount|currency|   status|order_date|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
|0ce692f2-13ca-46f...|2025-11-08 11:59:52|  CUST-9506|     US|757.76|     USD|CANCELLED|2025-11-08|
|d2916319-771d-40e...|2025-11-08 14:31:23|  CUST-8980|     US|124.97|     USD|CANCELLED|2025-11-08|
|9b05daae-6fc3-45a...|2025-11-08 18:33:42|  CUST-5230|     US|436.96|     USD|CANCELLED|2025-11-08|
|035eaa3c-2aa2-4cb...|2025-11-08 09:11:25|  CUST-1073|     US|652.49|     USD|CANCELLED|2025-11-08|
|20002243-fbae-4b3...|2025-11-08 05:23:42|  CUST-8488|     US|292.48|     USD|CANCELLED|2025-11-08|
+--------------------+-------------------+-----------+-------+------+--------+---------+----------+
only showing top 5 rows


7. Demonstrate Schema Evolution:

o Add payment_method &amp; coupon_code to new data.

o Write to the same Delta table, allowing schema evolution.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from datetime import datetime
import uuid

new_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True),   # NEW
    StructField("coupon_code", StringType(), True)        # NEW
])

new_data = [
    (
        str(uuid.uuid4()),
        datetime(2025, 10, 20, 14, 15),
        "CUST-5010",
        "US",
        220.75,
        "USD",
        "PAID",
        "CARD",
        "WELCOME10"
    ),
    (
        str(uuid.uuid4()),
        datetime(2025, 10, 21, 17, 40),
        "CUST-6135",
        "IN",
        499.99,
        "INR",
        "CREATED",
        "UPI",
        None
    )
]

df_new_orders = spark.createDataFrame(new_data, new_schema)

df_new_orders.show()
df_new_orders.printSchema()

+--------------------+-------------------+-----------+-------+------+--------+-------+--------------+-----------+
|            order_id|    order_timestamp|customer_id|country|amount|currency| status|payment_method|coupon_code|
+--------------------+-------------------+-----------+-------+------+--------+-------+--------------+-----------+
|f9a792ea-a808-432...|2025-10-20 14:15:00|  CUST-5010|     US|220.75|     USD|   PAID|          CARD|  WELCOME10|
|24eb15c2-3dd8-4e7...|2025-10-21 17:40:00|  CUST-6135|     IN|499.99|     INR|CREATED|           UPI|       NULL|
+--------------------+-------------------+-----------+-------+------+--------+-------+--------------+-----------+

root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: s

In [0]:
from pyspark.sql.functions import to_date
df_new_orders = df_new_orders.withColumn("order_date", to_date("order_timestamp"))

In [0]:
output_path = "/Volumes/main/default/orders_delta"

(
    df_new_orders.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")      # ENABLE SCHEMA EVOLUTION
        .partitionBy("country", "order_date")
        .save(output_path)
)

print("Schema evolution completed: new columns added!")

Schema evolution completed: new columns added!


In [0]:
df_check = spark.read.format("delta").load(output_path)
df_check.printSchema()
df_check.show(10, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- coupon_code: string (nullable = true)

+------------------------------------+-------------------+-----------+-------+------+--------+---------+----------+--------------+-----------+
|order_id                            |order_timestamp    |customer_id|country|amount|currency|status   |order_date|payment_method|coupon_code|
+------------------------------------+-------------------+-----------+-------+------+--------+---------+----------+--------------+-----------+
|9e27c3b1-dc1d-4f33-a2d0-3f0580e99fc0|2025-10-09 13:52:06|CUST-8880  |JP     |296.96|JPY     |CREATED  |2025-10-09|NULL          |NULL       |
|ac73

8. Demonstrate Updates &amp; Deletes using Delta:

o Mark some orders as CANCELLED.

o Delete orders below a certain amount (e.g., test data cleanup).

In [0]:
table_path = "/Volumes/main/default/orders_delta"

In [0]:
#Mark all orders below ₹100 (or $100 etc.) as CANCELLED
delta_tbl = DeltaTable.forPath(spark, table_path)

# Update: mark small orders as CANCELLED
delta_tbl.update(
    condition="amount < 100",
    set={"status": "'CANCELLED'"}
)

print("Updated orders where amount < 100")


Updated orders where amount < 100


In [0]:
#Delete test data — remove orders less than ₹50
delta_tbl.delete("amount < 50")

print("Deleted orders where amount < 50")

Deleted orders where amount < 50


In [0]:
#Verify Updates & Deletes
df_check = spark.read.format("delta").load(table_path)

df_check.groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|CANCELLED| 2620|
|  CREATED| 1121|
|     PAID| 5669|
+---------+-----+



In [0]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, table_path).history().show(20, truncate=False)

+-------+-------------------+--------------+-------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

9. (Bonus) Optimize the table:

o Use OPTIMIZE and optionally ZORDER on customer_id or order_date.

In [0]:
#Basic OPTIMIZE
spark.sql("""
OPTIMIZE delta.`/Volumes/main/default/orders_delta`
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
#OPTIMIZE with ZORDER
spark.sql("""
OPTIMIZE delta.`/Volumes/main/default/orders_delta`
ZORDER BY (customer_id)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
#File Count Before & After OPTIMIZE
dbutils.fs.ls("/Volumes/main/default/orders_delta")


[FileInfo(path='dbfs:/Volumes/main/default/orders_delta/_delta_log/', name='_delta_log/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=AU/', name='country=AU/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=DE/', name='country=DE/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=IN/', name='country=IN/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=JP/', name='country=JP/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=SG/', name='country=SG/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=UK/', name='country=UK/', size=0, modificationTime=1764613111540),
 FileInfo(path='dbfs:/Volumes/main/default/orders_delta/country=US/', name='country

In [0]:
df = spark.read.format("delta").load("/Volumes/main/default/orders_delta")
df.filter("customer_id = 'CUST-1234'").explain(True)

== Parsed Logical Plan ==
'Filter ('customer_id = CUST-1234)
+- Relation [order_id#15905,order_timestamp#15906,customer_id#15907,country#15908,amount#15909,currency#15910,status#15911,order_date#15912,payment_method#15913,coupon_code#15914] parquet

== Analyzed Logical Plan ==
order_id: string, order_timestamp: timestamp, customer_id: string, country: string, amount: double, currency: string, status: string, order_date: date, payment_method: string, coupon_code: string
Filter (customer_id#15907 = CUST-1234)
+- Relation [order_id#15905,order_timestamp#15906,customer_id#15907,country#15908,amount#15909,currency#15910,status#15911,order_date#15912,payment_method#15913,coupon_code#15914] parquet

== Optimized Logical Plan ==
Filter (isnotnull(customer_id#15907) AND (customer_id#15907 = CUST-1234))
+- Relation [order_id#15905,order_timestamp#15906,customer_id#15907,country#15908,amount#15909,currency#15910,status#15911,order_date#15912,payment_method#15913,coupon_code#15914] parquet

== Phy

10. (Bonus) Show how small file problems can occur with too many partitions and how
OPTIMIZE helps.

In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()


smallfile_demo_path = "/Volumes/main/default/orders_smallfiles"


for batch in range(120):            # write 120 tiny batches
    mini_df = (
        df_orders
            .orderBy("order_id")   
            .limit(40)              
            .repartition(4)         
    )

    (mini_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .save(smallfile_demo_path)
    )

In [0]:
def count_delta_files(folder):
    file_count = 0
    items = dbutils.fs.ls(folder)

    for obj in items:
        if obj.size > 0 and obj.path.endswith(".parquet"):   
            file_count += 1
        if obj.isDir():
            file_count += count_delta_files(obj.path)        

    return file_count

before_opt = count_delta_files(smallfile_demo_path)
print("Number of data files BEFORE OPTIMIZE:", before_opt)

Number of data files BEFORE OPTIMIZE: 480


In [0]:
%sql
OPTIMIZE delta.`/Volumes/main/default/orders_smallfiles`;

path,metrics
dbfs:/Volumes/main/default/orders_smallfiles,"List(1, 480, List(6446, 6446, 6446.0, 1, 6446), List(3012, 3040, 3023.75, 480, 1451400), 0, null, null, 0, 1, 480, 0, true, 0, 0, 1764614121359, 1764614126548, 8, 1, null, List(0, 0), null, 8, 8, 1424, 0, null)"


In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/main/default/orders_smallfiles`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
120,2025-12-01T18:35:27.000Z,72297342688245,mishamanmai2004@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,119,SnapshotIsolation,false,"Map(numRemovedFiles -> 480, numRemovedBytes -> 1451400, p25FileSize -> 6446, numDeletionVectorsRemoved -> 0, minFileSize -> 6446, numAddedFiles -> 1, maxFileSize -> 6446, p75FileSize -> 6446, p50FileSize -> 6446, numAddedBytes -> 6446)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
119,2025-12-01T18:34:26.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,118,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
118,2025-12-01T18:34:25.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,117,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
117,2025-12-01T18:34:24.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,116,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
116,2025-12-01T18:34:23.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,115,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
115,2025-12-01T18:34:21.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,114,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
114,2025-12-01T18:34:20.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,113,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
113,2025-12-01T18:34:19.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,112,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
112,2025-12-01T18:34:18.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,111,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
111,2025-12-01T18:34:16.000Z,72297342688245,mishamanmai2004@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3549743875246401),1201-174827-5wy9p9nu-v2n,110,WriteSerializable,true,"Map(numFiles -> 4, numOutputRows -> 40, numOutputBytes -> 12095)",null,Databricks-Runtime/17.2.x-aarch64-photon-scala2.13
